In [1]:
import pandas as pd
import nltk
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb


In [ ]:
#nltk.download('stopwords')

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("karthickveerakumar/spam-filter")

# print("Path to dataset files:", path)

### Exploring the data

In [2]:
df=pd.read_csv('emails.csv')

In [3]:
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [4]:
df['text'][0]

"Subject: naturally irresistible your corporate identity  lt is really hard to recollect a company : the  market is full of suqgestions and the information isoverwhelminq ; but a good  catchy logo , stylish statlonery and outstanding website  will make the task much easier .  we do not promise that havinq ordered a iogo your  company will automaticaily become a world ieader : it isguite ciear that  without good products , effective business organization and practicable aim it  will be hotat nowadays market ; but we do promise that your marketing efforts  will become much more effective . here is the list of clear  benefits : creativeness : hand - made , original logos , specially done  to reflect your distinctive company image . convenience : logo and stationery  are provided in all formats ; easy - to - use content management system letsyou  change your website content and even its structure . promptness : you  will see logo drafts within three business days . affordability : your  ma

In [5]:
df.spam.value_counts()

spam
0    4360
1    1368
Name: count, dtype: int64

### Text preprocessing & cleaning

In [6]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    # Initialize the stemmer
    stemmer = PorterStemmer()
    sentence = []
    text = text.lower()
    for w in text.split():
        if w=="subject:":
            continue
        # remove punctuation everywhere, not just at edges
        w = re.sub(r'[:\-_,.""]', '', w)
        if w:
            sentence.append(w)
            
    sentence = [w for w in sentence if w not in stop_words]
    
    sentence = [stemmer.stem(w) for w in sentence]
    
       
    return " ".join(sentence)


In [7]:
df['text']=df['text'].apply(clean_text)

In [8]:
df['text']

0       natur irresist corpor ident lt realli hard rec...
1       stock trade gunsling fanni merril muzo colza a...
2       unbeliev new home made easi im want show homeo...
3       4 color print special request addit inform ! c...
4       money get softwar cd ! softwar compat ' great ...
                              ...                        
5723    research develop charg gpg ! forward shirley c...
5724    receipt visit jim thank invit visit lsu shirle...
5725    enron case studi updat wow ! day ' super thank...
5726    interest david pleas call shirley crenshaw ( a...
5727    news aurora 5 2 updat aurora version 5 2 faste...
Name: text, Length: 5728, dtype: object

In [9]:
df.drop_duplicates(keep='last',inplace=True)

### Model building & choosing

In [10]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["spam"], test_size=0.2, random_state=42)

In [9]:
vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words="english")  

In [10]:
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [11]:
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

In [12]:
model = MultinomialNB().fit(X_train_vec, y_train, sample_weight=sample_weights)
Y_pred = model.predict(X_test_vec)
c_matrix = confusion_matrix(y_test, Y_pred)
print(c_matrix)
print(classification_report(y_test, Y_pred, target_names = ['Ham', 'Spam']))

[[808   5]
 [  5 289]]
              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       813
        Spam       0.98      0.98      0.98       294

    accuracy                           0.99      1107
   macro avg       0.99      0.99      0.99      1107
weighted avg       0.99      0.99      0.99      1107



In [13]:
# Define pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),   # Step 1: text → TF-IDF features
    ("clf", MultinomialNB())        # Step 2: placeholder classifier
])

# Define parameter grid
param_grid = [
    {
        "clf": [MultinomialNB()],
        "clf__alpha": [0.1, 1.0, 5.0]  # smoothing
    },
    {
        "clf": [LogisticRegression(max_iter=500)],
        "clf__C": [0.1, 1, 10],
        "clf__class_weight": [None, "balanced"]
    },
    {
        "clf": [LinearSVC(max_iter=2000)],
        "clf__C": [0.1, 1, 10],
        "clf__class_weight": [None, "balanced"]
    },
    {
        "clf": [xgb.XGBClassifier(
            use_label_encoder=False, eval_metric="logloss"
        )],
        "clf__scale_pos_weight": [1, 2, 5],   # handles imbalance
        "clf__max_depth": [3, 6],
        "clf__n_estimators": [100, 300]
    }
]

# Run grid search
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1_macro",   # better than accuracy for imbalance
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best model:", grid.best_estimator_)
print("Best params:", grid.best_params_)
print("CV Score:", grid.best_score_)


D:\Moneyfellows\credit_engine_env\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")
D:\Moneyfellows\credit_engine_env\lib\site-packages\numpy\ma\core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best model: Pipeline(steps=[('tfidf', TfidfVectorizer()),
                ('clf',
                 LinearSVC(C=1, class_weight='balanced', max_iter=2000))])
Best params: {'clf': LinearSVC(max_iter=2000), 'clf__C': 1, 'clf__class_weight': 'balanced'}
CV Score: 0.9895307882144284


<4582x317585 sparse matrix of type '<class 'numpy.float64'>'
	with 955937 stored elements in Compressed Sparse Row format>

### Enhance the model using semantic word vectorization

In [35]:
#pip install sentence-transformers
#pip install tf-keras
#pip install -U transformers sentence-transformers
#pip uninstall keras -y
#pip install tf-keras

In [25]:
from sentence_transformers import SentenceTransformer

# Load pretrained model
embedder = SentenceTransformer('all-MiniLM-L6-v2')  # small but powerful

# Encode all messages
X_embeddings = embedder.encode(X_train.tolist(), show_progress_bar=True)


Batches:   0%|          | 0/139 [00:00<?, ?it/s]

In [26]:
clf = LinearSVC(C=1).fit(X_embeddings, y_train, sample_weight=sample_weights)

In [32]:
clf2=xgb.XGBClassifier(eval_metric="logloss",
                       scale_pos_weight= len(df[df['spam']==0])/len(df[df['spam']==1]),
                       objective='binary:logistic',
                      eta = 0.2835965640598995,
                      gamma = 4.7790863261830365,
                      max_delta_step =  8,
                      max_depth = 3,
                      min_child_weight = 2,
                      num_round = 777
        )
clf2=clf2.fit(X_embeddings, y_train)
       

[22:00:39] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "num_round" } are not used.



In [27]:
new_vecs = embedder.encode(X_test.tolist())
preds = clf.predict(new_vecs)

In [28]:
c_matrix = confusion_matrix(y_test, preds)
print(c_matrix)
print(classification_report(y_test, preds, target_names = ['Ham', 'Spam']))

[[792  21]
 [ 14 280]]
              precision    recall  f1-score   support

         Ham       0.98      0.97      0.98       813
        Spam       0.93      0.95      0.94       294

    accuracy                           0.97      1107
   macro avg       0.96      0.96      0.96      1107
weighted avg       0.97      0.97      0.97      1107



In [33]:
preds2 = clf2.predict(new_vecs)

In [34]:
c_matrix2 = confusion_matrix(y_test, preds2)
print(c_matrix2)
print(classification_report(y_test, preds, target_names = ['Ham', 'Spam']))

[[791  22]
 [ 21 273]]
              precision    recall  f1-score   support

         Ham       0.98      0.97      0.98       813
        Spam       0.93      0.95      0.94       294

    accuracy                           0.97      1107
   macro avg       0.96      0.96      0.96      1107
weighted avg       0.97      0.97      0.97      1107



In [41]:
clf2.predict(embedder.encode(['you will receive this money if you click on this link']))

array([0])

In [40]:
clf.predict(embedder.encode(['you will receive this money if you click on this link']))

array([1], dtype=int64)